# Эксперимент 1: SGD и расписания шага

Цель: увидеть шумовую окрестность SGD с постоянным шагом и сравнить `constant`, `inverse_sqrt`, `step_decay` с полным градиентным спуском.

Ось X на графиках — эффективные эпохи: один полный проход по данным равен `m` индивидуальным градиентам.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
from src.experiments import make_team_oracles, plot_histories, hardware_description
from src.optimization import gradient_descent, sgd

FIGS = ROOT / 'figs'
FIGS.mkdir(exist_ok=True)

print(hardware_description())

In [ ]:
oracles = make_team_oracles(m=2000, n=30, regcoef=1e-2, random_state=42)
x0_by_oracle = {name: 0.1 * __import__('numpy').zeros(oracle.n) for name, oracle in oracles.items()}

for name, oracle in oracles.items():
    x0 = x0_by_oracle[name]
    histories = {}
    _, _, histories['GD'] = gradient_descent(oracle, x0, step_size=0.15, max_epoch=50, trace=True)
    _, _, histories['SGD constant'] = sgd(
        oracle, x0, batch_size=32, max_epoch=50,
        lr_schedule='constant', lr_params={'alpha_0': 0.03}, trace=True, random_state=1
    )
    _, _, histories['SGD inverse sqrt'] = sgd(
        oracle, x0, batch_size=32, max_epoch=50,
        lr_schedule='inverse_sqrt', lr_params={'alpha_0': 0.20}, trace=True, random_state=1
    )
    _, _, histories['SGD step decay'] = sgd(
        oracle, x0, batch_size=32, max_epoch=50,
        lr_schedule='step_decay', lr_params={'alpha_0': 0.08, 'gamma': 0.5, 'drop_freq': 10},
        trace=True, random_state=1
    )
    plot_histories(histories, y_key='func', title=f'{name}: F(x) от эффективных эпох')
    file_stem = name.lower().replace('-', '').replace(' ', '_')
    plt.savefig(FIGS / f'01_sgd_schedules_{file_stem}.png', dpi=200, bbox_inches='tight')

Постоянный шаг обычно быстро снижает функцию в начале, но затем остается в шумовой окрестности. Убывающие расписания чаще уходят ниже, однако теряют скорость на поздних эпохах.